In [ ]:
%pip install -q krauncher-jupyter

# Tutorial 20: fine-tune BERT on IMDB with `%%krauncher`

A real fine-tune, not a synthetic loop: `bert-base-uncased` on a subset of the
IMDB sentiment dataset via the HuggingFace `Trainer`. The marked cell is written
exactly the way a normal notebook uses the Hub — plain
`load_dataset("stanfordnlp/imdb")` and `from_pretrained("google-bert/bert-base-uncased")`.
The magic pre-fetches those literal Hub references before the container starts,
runs the cell on the cheapest suitable GPU, streams the `Trainer` progress live,
and hands the metrics back as notebook variables.

The dataset is subset to a few thousand reviews so the run finishes in minutes;
drop `train_size` / `eval_size` to use more.

In [ ]:
%load_ext krauncher_magic

In [ ]:
import os, getpass

os.environ["CAS_API_KEY"] = getpass.getpass("CAS_API_KEY: ")

### Training config

These notebook variables become the marked cell's inputs (`num_epochs`, `batch_size`, `lr`, `train_size`, `eval_size` are auto-detected as free variables).

In [ ]:
num_epochs = 2
batch_size = 16
lr = 2e-5
train_size = 2000
eval_size = 1000

### Fine-tune

Run the cell — the quote prints first, then live `Trainer` progress, then the returned metrics land back in the notebook.

In [ ]:
%%krauncher --pip transformers,datasets,accelerate --vram 6 --timeout 1200 --out eval_accuracy,train_loss,eval_loss,train_runtime_sec
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

# HF-native: these literal Hub refs are pre-fetched by the data bridge before
# the container starts, so the unmodified calls resolve from the local cache.
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained(
    "google-bert/bert-base-uncased", num_labels=2,
)

# IMDB splits are ordered by label — shuffle before slicing to keep both classes.
raw = load_dataset("stanfordnlp/imdb")
train_ds = raw["train"].shuffle(seed=42).select(range(train_size))
eval_ds = raw["test"].shuffle(seed=42).select(range(eval_size))

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=256)

train_ds = train_ds.map(tokenize, batched=True, batch_size=1000).rename_column("label", "labels")
eval_ds = eval_ds.map(tokenize, batched=True, batch_size=1000).rename_column("label", "labels")
train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
eval_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
print(f"Fine-tuning bert-base-uncased on {len(train_ds)} train / {len(eval_ds)} eval reviews")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float((preds == labels).mean())}

args = TrainingArguments(
    output_dir="/tmp/bert-imdb",
    num_train_epochs=num_epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size * 2,
    learning_rate=lr,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=20,
    fp16=True,
    report_to="none",
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
)
train_out = trainer.train()
eval_out = trainer.evaluate()

train_loss = round(train_out.training_loss, 4)
eval_accuracy = round(eval_out["eval_accuracy"], 4)
eval_loss = round(eval_out["eval_loss"], 4)
train_runtime_sec = round(train_out.metrics["train_runtime"], 1)

### Results

`eval_accuracy`, `train_loss`, `eval_loss` and `train_runtime_sec` are now ordinary notebook variables.

In [ ]:
print(f"Eval accuracy: {eval_accuracy:.2%}")
print(f"Train loss:    {train_loss}")
print(f"Eval loss:     {eval_loss}")
print(f"Train runtime: {train_runtime_sec:.0f}s")